---
title: "Exercise 3. MAGMA SNP Annotation"
subtitle: "Map QC-filtered SNPs to nearby genes using MAGMA inside the course container"
format:
  html:
    toc: true
    toc-depth: 3
    number-sections: true
jupyter: bash
---

# Overview

This notebook converts SNP coordinates into gene assignments. MAGMA does this by combining a SNP-location file from notebook 2 with a gene-location reference table. The result is a `.genes.annot` file that tells MAGMA which SNPs belong to which genes.

::: {.callout-note}
## Learning goals
By the end of this notebook, you should be able to:
- explain why MAGMA needs both SNP-location and gene-location inputs
- choose an annotation window and describe what biological assumptions it makes
- interpret the role of the `.genes.annot` output in the next MAGMA step

## Prerequisites
Run notebook 2 first so that `input/magma/snp_loc_full.txt` and `input/magma/snp_loc_demo.txt` exist locally. This notebook also assumes `../gwas-minimal.sif` exists and that `apptainer` is available on the system.
:::

::: {.callout-note}
## Questions to keep in mind

1. What biological assumptions are hidden inside a fixed annotation window such as 10 kb upstream and downstream?
2. How might annotation change if you used a different gene model or genome build?
3. Why is SNP-to-gene mapping already an interpretation step rather than a purely mechanical one?
:::

In [23]:
REF_DIR=reference_data
INPUT_DIR=input/magma
OUT_DIR=output/magma
GENE_LOC=${REF_DIR}/NCBI37.3/NCBI37.3.gene.loc
WINDOW=10,10

mkdir -p ${OUT_DIR}
if [ ! -d ${REF_DIR}/g1000_eur ]; then
    curl https://zenodo.org/records/14142263/files/g1000_eur.zip?download=1 -o ${REF_DIR}/g1000_eur.zip
    unzip ${REF_DIR}/g1000_eur.zip -d ${REF_DIR}/g1000_eur
else
    echo "Reference data already exists, skipping download and extraction."
fi

printf 'Gene location file: %s\n' "${GENE_LOC}"

Reference data already exists, skipping download and extraction.
Gene location file: reference_data/NCBI37.3/NCBI37.3.gene.loc


# Why annotation windows matter

MAGMA can assign SNPs only inside the gene body or include flanking windows. Here we use `10,10`, meaning 10 kb upstream and 10 kb downstream. This captures some nearby regulatory variation but still reflects a fairly simple positional model.

::: {.callout-warning}
A positional window does not capture all regulatory biology. Many causal variants act over longer distances or through chromatin interactions not represented here.
:::

In [24]:
run_annotation() {
  local label=$1
  local snp_loc=$2
  local out_prefix=$3

  echo
  echo '============================================================'
  echo "Running MAGMA SNP annotation: ${label}"
  echo '============================================================'
  echo "SNP location file: ${snp_loc}"
  echo "Gene location file: ${GENE_LOC}"
  echo "Window: ${WINDOW} kb"
  echo "Output prefix: ${out_prefix}"

    magma \
    --annotate window=${WINDOW} \
    --snp-loc ${snp_loc} \
    --gene-loc ${GENE_LOC} \
    --out ${out_prefix}
}

In [20]:
run_annotation FULL ${INPUT_DIR}/snp_loc_full.txt ${OUT_DIR}/adhd_full


Running MAGMA SNP annotation: FULL
SNP location file: input/magma/snp_loc_full.txt
Gene location file: reference_data/NCBI37.3/NCBI37.3.gene.loc
Window: 10,10 kb
Output prefix: output/magma/adhd_full
Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--annotate window=10,10
	--snp-loc input/magma/snp_loc_full.txt
	--gene-loc reference_data/NCBI37.3/NCBI37.3.gene.loc
	--out output/magma/adhd_full

Start time is 10:47:49, Tuesday 19 May 2026

Starting annotation...
Reading gene locations from file reference_data/NCBI37.3/NCBI37.3.gene.loc... 
	adding window: 10000bp
	19427 gene locations read from file
	chromosome  1: 2016 genes
	chromosome  2: 1226 genes
	chromosome  3: 1050 genes
	chromosome  4: 745 genes
	chromosome  5: 856 genes
	chromosome  6: 1016 genes
	chromosome  7: 906 genes
	chromosome  8: 669 genes
	chromosome  9: 775 genes
	chromosome 10: 723 genes
	chromosome 11: 1275 genes
	chromosome 12: 1009 genes
	chromosome 13: 320 genes
	chromosome 14: 595 genes
	chromosome 15: 586 genes

In [25]:
run_annotation DEMO ${INPUT_DIR}/snp_loc_demo.txt ${OUT_DIR}/adhd_demo


Running MAGMA SNP annotation: DEMO
SNP location file: input/magma/snp_loc_demo.txt
Gene location file: reference_data/NCBI37.3/NCBI37.3.gene.loc
Window: 10,10 kb
Output prefix: output/magma/adhd_demo
Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--annotate window=10,10
	--snp-loc input/magma/snp_loc_demo.txt
	--gene-loc reference_data/NCBI37.3/NCBI37.3.gene.loc
	--out output/magma/adhd_demo

Start time is 10:50:02, Tuesday 19 May 2026

Starting annotation...
Reading gene locations from file reference_data/NCBI37.3/NCBI37.3.gene.loc... 
	adding window: 10000bp
	19427 gene locations read from file
	chromosome  1: 2016 genes
	chromosome  2: 1226 genes
	chromosome  3: 1050 genes
	chromosome  4: 745 genes
	chromosome  5: 856 genes
	chromosome  6: 1016 genes
	chromosome  7: 906 genes
	chromosome  8: 669 genes
	chromosome  9: 775 genes
	chromosome 10: 723 genes
	chromosome 11: 1275 genes
	chromosome 12: 1009 genes
	chromosome 13: 320 genes
	chromosome 14: 595 genes
	chromosome 15: 586 genes

In [27]:
echo 'Produced annotation files:'
ls -lh ${OUT_DIR}/*.genes.annot | cut -d' ' -f9

Produced annotation files:
output/magma/adhd_demo.genes.annot
output/magma/adhd_full.genes.annot


# Interpretation and discussion

A gene annotation file is still not a gene-level association result. It is only the mapping layer that defines which SNPs contribute to each gene test in the next step.

::: {.callout-tip}
## Reflection prompts
1. Which genes are most sensitive to the choice of annotation window?
2. What kinds of true biology are likely to be missed by distance-only mapping?
3. If a gene has many SNPs assigned to it, does that automatically make it more significant later? Why not?
:::